# Full fine-tuning at 288 px (`09_full_288`)

Built from `01_full.ipynb`: the same model, method and training recipe, with larger images for training and testing. **This is a post-deadline experiment for the report.** It uses only the training set, with the same 20,000-image holdout as the other notebooks. It never reads the val or test sets and writes no submission files.

> The models trained here are only evaluated on the holdout and are not used for any submission. Our ranked Phase 2 result comes from the Phase 1 model `1019_1319_1_full`. `cs604_vit_288.ipynb` in the project root trains the final recipe at 288 px; run 1 here chose its test size (320 px). Its planned unranked test submission could not be made, so it has no test score.

**Why**: `08_linear_exp.ipynb` tested our trained 224 px model `1019_1120_1_full` at larger sizes, with no extra training. Its holdout top-1 went from 73.38% at 224 px to **75.50% at 288 px** (+2.13 points). This notebook measures how much more we get when the model is also trained at 288 px.

**Changes from 01**
- The model is created with `dynamic_img_size=True` (§4). timm then interpolates the 14 × 14 position embeddings to the patch grid of each input, e.g. 18 × 18 at 288 px. So the same model can train and test at any size that is a multiple of 16. At 224 px the model is exactly the one in 01.
- `TRAIN_SIZES` sets the training image size of each epoch (§1). The training data loader is rebuilt when the size changes (§7). The prototype head uses features at the size of the first epoch (§6).
- After each epoch, the holdout is evaluated at `EVAL_SIZE` (288 px). After training, it is evaluated at every size in `FINAL_EVAL_SIZES` (224 to 448 px), with the inference latency and throughput at each size (§8).

**Experiments** (§1 Config)

| Run | Training size of epochs 1, 2, 3 | Idea |
|---|---|---|
| `<batch>_1_full_288` | 288, 288, 288 | train and test at 288 px; its best test size sets `TEST_SIZE` in `cs604_vit_288.ipynb` |
| `<batch>_2_full_224to288` (optional, commented out) | 224, 224, 288 | cheaper: only the last epoch at 288 px, as in FixRes |
| Reference: `1019_1120_1_full` (01 defaults, tested in 08) | 224, 224, 224 | 73.38% at 224 px, 75.50% at 288 px |

**Cost** (estimates for an A100-80GB): a 288 px image has 325 tokens instead of 197, so it needs about 1.7× the compute. Run 1: about 0.7 GPU-hours and about 30 GB of memory (01 at 224 px: 0.48 GPU-hours, 18.9 GB). Run 2 (optional): about 0.55 GPU-hours. Both stay within the budget of 1.2 GPU-hours per run. With data preparation and the final evaluations, the notebook takes about 1 hour with run 1 only, or about 1.5 hours with both runs.

**How to run**: Run all cells on an A100. Results are on Drive: each run in `cs604/runs/<run_name>/`, and the tables `results.md` and `resolution.md` with the plot `resolution.png` in `cs604/batches/<batch_name>/`.

**How to read the results**: compare each run at 288 px with the reference at 288 px (75.50%); `results.md` shows this difference as "vs reference". `resolution.md` also shows which test size is best for each run. Set `TEST_SIZE` in `cs604_vit_288.ipynb` to the best test size of run 1 (the bold value in its row). The sweep goes up to 448 px, because after training at 288 px the best size may be well above 288 px, in the same way that 288 px was best for the model trained at 224 px. In earlier runs, the holdout was within about 0.3 points of the CodaBench score, so it is a fair estimate of what the test set would give.

## 1. Config

Each dict in `EXPERIMENTS` is one run and lists only the settings that differ from the defaults below. `TRAIN_SIZES` has one image size per epoch, so its length must equal `EPOCHS`. All sizes must be multiples of 16, the patch size. The other defaults are the comparison protocol shared by the method notebooks, the same as in 01.

In [ ]:
# ===== Experiments to run: each dict is one run and lists only the settings that differ from the defaults below =====
EXPERIMENTS = [
    dict(TRAIN_SIZES=[288, 288, 288]),    # train and test at 288 px; its best test size sets TEST_SIZE in cs604_vit_288.ipynb
    # dict(TRAIN_SIZES=[224, 224, 288]),  # optional, for the report: only the last epoch at 288 px (cheaper, as in FixRes)
    # dict(TRAIN_SIZES=[224, 224, 224]),  # a rerun of 1019_1120_1_full, to see the run-to-run noise (about ±0.3 points)
]

# ===== Image sizes in pixels, all multiples of 16 (the patch size) =====
TRAIN_SIZES = [288, 288, 288]  # training size of each epoch, one entry per epoch
EVAL_SIZE = 288                # holdout size after each epoch; a run's main result is measured at this size
FINAL_EVAL_SIZES = [224, 256, 288, 320, 352, 384, 416, 448]  # after training, holdout accuracy and latency at each size

# ===== Reference: 1019_1120_1_full (01's defaults, trained at 224 px), tested at each size by 08_linear_exp (batch 1105_0141) =====
REFERENCE_RUN = "1019_1120_1_full"
REFERENCE = {224: 0.73375, 256: 0.7521, 288: 0.75505, 320: 0.7528, 352: 0.7433, 384: 0.7294}  # holdout top-1 at each test size

# ===== Defaults: the comparison protocol shared by all method notebooks (prototype head, 3 epochs, 2 holdout images per class) =====
METHOD = "full"
MODEL_NAME = "vit_base_patch16_224.augreg2_in21k_ft_in1k"  # timm name of the base model in the doc (IN21K→IN1K)
HOLDOUT_PER_CLASS = 2    # images per class held out from train for local evaluation (the official val set has no labels)
EPOCHS = 3
WARMUP_EPOCHS = 0.3      # the LR rises linearly for this many epochs, then follows a cosine down to 0
BATCH_SIZE = None        # images per step; None = pick based on VRAM (256 on a 40 or 80 GB GPU)
EFFECTIVE_BATCH = 256    # if one step can't fit it, gradients are accumulated over several steps to reach it
LR = 1e-4
WEIGHT_DECAY = 0.05
LAYER_DECAY = 0.75       # layer-wise LR decay: the last block gets LR × 0.75, the first block LR × 0.75^12
DROP_PATH = 0.1          # stochastic depth: residual branches are randomly skipped during training
LABEL_SMOOTHING = 0.1
RRC_SCALE = (0.25, 1.0)  # area range for RandomResizedCrop (training augmentation)
LOG_EVERY = 100          # write a log line every N optimizer steps
SEED = 0
NUM_CLASSES = 10000

# ===== Data and paths =====
HF_REPO = "doem1997/cs604-course-data"       # the course dataset on Hugging Face
DATA_FROM_HF = True      # True: download the data from Hugging Face (about 500 MB/s on Colab); False: use the copy on Drive (about 33 MB/s)
DRIVE_DIR = "/content/drive/MyDrive/cs604"   # results are saved here
DRIVE_DATA_DIR = f"{DRIVE_DIR}/data"         # copy of the dataset on Drive, used only when DATA_FROM_HF = False (made by cs604_download_to_drive.ipynb)
DATA_DIR = "/content/data"                   # each session puts the data on the local Colab disk; training reads images from here
DISCONNECT_WHEN_DONE = False  # release the runtime when done to stop using compute units (handy when running in the background)

## 2. Setup

Install the libraries, mount Google Drive, check the GPU (it must support bf16: A100, L4 or H100), and set up the log. All outputs are saved on Drive, so nothing is lost if the session disconnects.

In [ ]:
# Colab already has torch / torchvision; install timm here, and hf_xet for fast parallel downloads from Hugging Face
!pip install -q timm hf_xet

In [ ]:
# Settings an experiment in EXPERIMENTS may override
RUN_KEYS = ("MODEL_NAME", "HOLDOUT_PER_CLASS", "EPOCHS", "WARMUP_EPOCHS", "BATCH_SIZE", "EFFECTIVE_BATCH",
            "LR", "WEIGHT_DECAY", "LAYER_DECAY", "DROP_PATH", "LABEL_SMOOTHING", "RRC_SCALE", "LOG_EVERY",
            "SEED", "NUM_CLASSES", "TRAIN_SIZES", "EVAL_SIZE", "FINAL_EVAL_SIZES")
BASE_CONFIG = {k: globals()[k] for k in RUN_KEYS}

import os, re, gc, json, math, time, random, shutil, subprocess
from datetime import datetime
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from google.colab import drive

# ---- Mount Google Drive: data is read from here, and results are saved here, so nothing is lost on disconnect ----
drive.mount("/content/drive")
OUT_ROOT = DRIVE_DIR

# ---- GPU: needs bf16 support, i.e. an A100, L4 or H100 runtime ----
assert torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8, (
    "Pick a GPU runtime with bf16 support: A100, L4 or H100")
DEVICE = "cuda"
props = torch.cuda.get_device_properties(0)
GPU_NAME = props.name
gib = props.total_memory / 1024**3
VRAM_GB = next((v for v in (8, 12, 16, 24, 32, 40, 48, 80, 94, 96, 141, 192) if v >= 0.98 * gib), round(gib))  # nominal VRAM
torch.backends.cuda.matmul.allow_tf32 = True  # TF32 for float32 matrix multiplies: much faster, accuracy barely changes
torch.backends.cudnn.allow_tf32 = True
NUM_WORKERS = min(16, os.cpu_count())  # processes that read and decode images in parallel
VERSIONS = f"torch {torch.__version__} (CUDA {torch.version.cuda}), timm {timm.__version__}"

os.environ["TZ"] = "Asia/Singapore"  # the Colab VM runs on UTC; use local time for batch names and log timestamps
time.tzset()
BATCH_NAME = datetime.now().strftime("%m%d_%H%M")  # this session's runs are named <BATCH_NAME>_<i>_<method>
BATCH_DIR = os.path.join(OUT_ROOT, "batches", BATCH_NAME)
os.makedirs(BATCH_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

LOG_FILE = None  # train_log.txt of the current run, set by run_experiment


def log(*args):
    """Print, and also append to this batch's batch_log.txt and the current run's train_log.txt (the report needs the training log)"""
    msg = " ".join(str(a) for a in args)
    print(msg, flush=True)
    stamp = datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ")
    for path in (os.path.join(BATCH_DIR, "batch_log.txt"), LOG_FILE):
        if path:
            with open(path, "a") as f:
                f.write(stamp + msg + "\n")


ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
log(f"Batch {BATCH_NAME} -> {BATCH_DIR}")
log(f"Device {GPU_NAME} ({VRAM_GB} GB), bfloat16, {os.cpu_count()} CPUs, {ram_gb:.0f} GB RAM, "
    f"{shutil.disk_usage(DATA_DIR).free / 1e9:.0f} GB free disk")
log(f"Versions: {VERSIONS}")

## 3. Prepare the data

- In each new session, the data is first copied to the Colab local disk (`DATA_DIR`), and the images are then read from there. Reading 450,000 small images one by one from the internet or from Drive is very slow.
- By default the data comes from Hugging Face (`DATA_FROM_HF = True`). On Colab this runs at about 495 MB/s, so the 40 GB `train.tar.gz` downloads in just over a minute. The archive is deleted after it is unpacked. During download and unpacking, the local disk needs about 80 GB of free space.
- With `DATA_FROM_HF = False`, the backup on Drive is used instead (`DRIVE_DATA_DIR`; save it there first with `cs604_download_to_drive.ipynb`). It is unpacked directly from Drive at about 33 MB/s, which takes about 20 minutes.
- Images keep their original size (longest side 500 pixels) and are not resized in advance: on an A100 runtime, the CPU decodes fast enough to keep up with the GPU.
- When the data is ready, a marker file is written, so this step is skipped if you run it again in the same session.
- Only the training set is used here. The official val set has no labels, and the submission file is only written in the main notebook.

In [ ]:
def prepare_split(split):
    """Put {split}.txt and the images of {split}.tar.gz on the local Colab disk under DATA_DIR. The files come from Hugging Face,
    or, with DATA_FROM_HF = False, from the copy on Drive. Returns the path of the local {split}.txt"""
    list_path = os.path.join(DATA_DIR, f"{split}.txt")
    marker = os.path.join(DATA_DIR, f".{split}_ready")
    if os.path.exists(marker):  # already done in this session
        return list_path
    t0 = time.time()
    if DATA_FROM_HF:
        hf_hub_download(HF_REPO, f"{split}.txt", repo_type="dataset", local_dir=DATA_DIR)
        tar_path = hf_hub_download(HF_REPO, f"{split}.tar.gz", repo_type="dataset", local_dir=os.path.join(DATA_DIR, "hf"))
        log(f"[{split}] downloaded from Hugging Face in {(time.time() - t0) / 60:.1f} min")
    else:
        shutil.copy(os.path.join(DRIVE_DATA_DIR, f"{split}.txt"), list_path)
        tar_path = os.path.join(DRIVE_DATA_DIR, f"{split}.tar.gz")
    log(f"[{split}] extracting {tar_path} ...")
    subprocess.run(["tar", "-xzf", tar_path, "-C", DATA_DIR], check=True)
    if DATA_FROM_HF:
        os.remove(tar_path)  # the images are out; free the disk space the archive takes (40 GB for train)
    with open(list_path) as f:
        n_expected = sum(1 for line in f if line.strip())
    n_found = sum(len(files) for _, _, files in os.walk(os.path.join(DATA_DIR, split)))
    assert n_found == n_expected, f"{split}: found {n_found} images, expected {n_expected}"
    open(marker, "w").close()
    log(f"[{split}] {n_found} images ready in {(time.time() - t0) / 60:.1f} min")
    return list_path


def read_list(path):
    """Each line is '<image path> <class id>', e.g. 'train/class_0000/9aef28bf-....jpg 0'"""
    items = []
    with open(path) as f:
        for line in f:
            parts = line.split()
            if parts:
                items.append((parts[0], int(parts[1])))
    return items


TRAIN_ALL = read_list(prepare_split("train"))
log(f"train {len(TRAIN_ALL):,} images | {NUM_WORKERS} loader workers")

## 4. Model and the full fine-tuning method

- `create_model` loads the pretrained ViT-B/16 with `dynamic_img_size=True`, replaces the old 1,000-class head with a new 10,000-class head, and sets drop path. `dynamic_img_size=True` is the only change from 01 in this section.
- With it, the stored 14 × 14 position embeddings (one per patch at 224 px) are resized by bicubic interpolation to the patch grid of each input, in every forward pass: 16 × 16 at 256 px, 18 × 18 at 288 px, 24 × 24 at 384 px. The stored embeddings stay 14 × 14 and are trained through the interpolation. At 224 px they are used as they are, so the model is exactly the one in 01.
- `apply_method` is just one line: it makes all parameters trainable. Parameters: 85,798,656 in the backbone (100%), plus 7,690,000 in the head. The image size does not change the number of parameters.
- `resolve_config` checks that `TRAIN_SIZES` has one size per epoch and that all sizes are multiples of 16.

In [ ]:
def is_head(name):
    return name in ("head.weight", "head.bias")


def resolve_config(overrides):
    """Defaults from section 1 + one experiment's overrides → the full config of this run"""
    unknown = set(overrides) - set(RUN_KEYS)
    assert not unknown, f"unknown keys in experiment config: {sorted(unknown)}"
    cfg = SimpleNamespace(**{**BASE_CONFIG, **overrides})
    assert cfg.HOLDOUT_PER_CLASS > 0, "this notebook evaluates on the holdout set, so keep at least 1 image per class"
    assert len(cfg.TRAIN_SIZES) == cfg.EPOCHS, f"TRAIN_SIZES needs one size per epoch: {cfg.TRAIN_SIZES} for {cfg.EPOCHS} epochs"
    sizes = [*cfg.TRAIN_SIZES, cfg.EVAL_SIZE, *cfg.FINAL_EVAL_SIZES]
    assert all(s % 16 == 0 for s in sizes), f"image sizes must be multiples of the 16 px patch size: {sizes}"
    if cfg.BATCH_SIZE is None:
        cfg.BATCH_SIZE = 256 if VRAM_GB >= 40 else 128 if VRAM_GB >= 24 else 64
    cfg.ACCUM_STEPS = max(1, cfg.EFFECTIVE_BATCH // cfg.BATCH_SIZE)
    return cfg


def create_model(cfg):
    """Pretrained ViT-B/16 from timm; its 1000-class ImageNet head is replaced by a new, randomly initialized 10,000-class head.
    dynamic_img_size=True: in every forward pass, the 14 × 14 position embeddings are interpolated to the patch grid of the
    input (18 × 18 at 288 px), so the model takes any size that is a multiple of 16. At 224 px they are used as they are"""
    return timm.create_model(cfg.MODEL_NAME, pretrained=True, num_classes=cfg.NUM_CLASSES,
                             drop_path_rate=cfg.DROP_PATH, dynamic_img_size=True).to(DEVICE)


def apply_method(net, cfg):
    """Full fine-tuning: every parameter of the backbone and the new head is trained"""
    return net.requires_grad_(True)


def param_report(net, ctx):
    """Trainable parameter counts. The 7.69M head is counted separately, as in the doc's LoRA reference (0.34%)"""
    n_train = sum(p.numel() for n, p in net.named_parameters() if p.requires_grad and not is_head(n))
    return {"backbone_params": ctx.n_backbone, "head_params": ctx.n_head, "head_mode": "fine-tuned",
            "trainable_excl_head": n_train, "trainable_pct": 100 * n_train / ctx.n_backbone,
            "trainable_incl_head": n_train + ctx.n_head}

## 5. Dataset and local holdout set

The official val set has no labels, so we keep `HOLDOUT_PER_CLASS` images per class from the training set as a local holdout set. It is only used to measure accuracy, never for training. The split is fixed by `SEED`, so it is the same in every experiment: the same 20,000 images as in notebooks 01 to 08.

The image transforms now take the size as an argument (`make_transform`):
- Training: a random crop (area range `RRC_SCALE`) resized to size × size, and a random flip.
- Evaluation: the short side is resized to size / 0.9, then the center size × size is cropped. 0.9 is the crop ratio of the pretrained model; at 288 px the image is resized to 320 and cropped to 288.

In [ ]:
def split_holdout(items, k, seed):
    """Hold out k images of every class: shuffle each class's image list with a fixed seed and take the first k"""
    by_class = {}
    for path, label in items:
        by_class.setdefault(label, []).append(path)
    rng = random.Random(seed)
    train, holdout = [], []
    for label in sorted(by_class):
        paths = sorted(by_class[label])
        rng.shuffle(paths)
        holdout += [(p, label) for p in paths[:k]]
        train += [(p, label) for p in paths[k:]]
    return train, holdout


class ImageList(Dataset):
    def __init__(self, items, transform):
        self.items, self.transform = items, transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label = self.items[i]
        with Image.open(os.path.join(DATA_DIR, path)) as im:
            img = im.convert("RGB")
        return self.transform(img), label


BICUBIC = T.InterpolationMode.BICUBIC


def make_ctx(model, cfg):
    """Data setup for this run: train/holdout split, normalization values, the crop ratio for evaluation, param counts"""
    data_cfg = timm.data.resolve_model_data_config(model)
    train_items, holdout_items = split_holdout(TRAIN_ALL, cfg.HOLDOUT_PER_CLASS, cfg.SEED)
    return SimpleNamespace(
        train_items=train_items, holdout_items=holdout_items, crop_pct=data_cfg["crop_pct"],
        mean=torch.tensor(data_cfg["mean"], device=DEVICE).view(1, 3, 1, 1) * 255,
        std=torch.tensor(data_cfg["std"], device=DEVICE).view(1, 3, 1, 1) * 255,
        n_backbone=sum(p.numel() for n, p in model.named_parameters() if not is_head(n)),
        n_head=sum(p.numel() for p in model.head.parameters()),
    )


def make_transform(size, train, ctx, cfg):
    """Training: random crop resized to size × size, random flip. Evaluation: resize the short side to size / crop_pct,
    then crop the center size × size. Only crop/flip on the CPU and output uint8; normalize on the GPU to save CPU work"""
    if train:
        return T.Compose([T.RandomResizedCrop(size, scale=tuple(cfg.RRC_SCALE), interpolation=BICUBIC),
                          T.RandomHorizontalFlip(), T.PILToTensor()])
    return T.Compose([T.Resize(int(size / ctx.crop_pct), interpolation=BICUBIC), T.CenterCrop(size), T.PILToTensor()])


class Loader(DataLoader):
    """Before starting new worker processes, run garbage collection in the main process. Otherwise an old, interrupted DataLoader is copied into the new workers,
    and when it is collected there it prints lots of harmless AssertionError: can only test a child process messages."""

    def __iter__(self):
        gc.collect()
        return super().__iter__()


def make_loader(items, train, size, ctx, cfg):
    """Images of size × size px. Training: random crops, shuffled, BATCH_SIZE images per step.
    Evaluation: center crops, in order, 2 × BATCH_SIZE (no gradients, so it fits)"""
    return Loader(ImageList(items, make_transform(size, train, ctx, cfg)),
                  batch_size=cfg.BATCH_SIZE if train else 2 * cfg.BATCH_SIZE, shuffle=train, drop_last=train,
                  num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=train)


def to_input(x, ctx):
    """uint8 images (B, 3, size, size) → normalized float32 on the GPU"""
    return (x.to(DEVICE, non_blocking=True).float() - ctx.mean) / ctx.std


def autocast():
    """bf16 mixed precision: matrix multiplies run in bfloat16, the weights stay float32"""
    return torch.autocast("cuda", dtype=torch.bfloat16)


def free_gpu():
    """Free the GPU memory left by the previous run and reset the peak memory stats"""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## 6. Classifier head initialization

If the new 10,000-class head starts from random weights, the gradients early in training will damage the pretrained features. So we first use the frozen backbone to extract features for all training images. For each class, the average direction of its features becomes the weight of that class (the class prototype), multiplied by a temperature. The time for this step, including feature extraction, counts toward the GPU time of the experiment. Here the features are extracted at the size of the first epoch (`TRAIN_SIZES[0]`): 288 px for run 1 and 224 px for run 2.

In [ ]:
class Clock:
    """Total GPU work time of a run: head init + training + holdout evaluation during training (not data prep)"""

    def __init__(self):
        self.used, self._t0 = 0.0, None

    def __enter__(self):
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        torch.cuda.synchronize()
        self.used += time.time() - self._t0
        self._t0 = None

    @property
    def elapsed(self):
        return self.used + (time.time() - self._t0 if self._t0 else 0.0)


@torch.inference_mode()
def extract_features(net, items, size, ctx, cfg):
    """Features of the frozen backbone at size × size px: the class-token output after the final LayerNorm, 768 numbers per image"""
    net.eval()
    feats, labels = [], []
    for x, y in tqdm(make_loader(items, False, size, ctx, cfg), desc="train features", leave=False):
        with autocast():
            f = net.forward_head(net.forward_features(to_input(x, ctx)), pre_logits=True)
        feats.append(f.half())
        labels.append(y.to(DEVICE))
    return torch.cat(feats), torch.cat(labels)


@torch.inference_mode()
def class_prototypes(feats, labels, num_classes):
    """Mean direction of each class: L2-normalize every feature, add them up per class, normalize the sums"""
    protos = torch.zeros(num_classes, feats.shape[1], device=feats.device)
    for i in range(0, len(feats), 65536):
        protos.index_add_(0, labels[i:i + 65536], F.normalize(feats[i:i + 65536].float(), dim=1))
    return F.normalize(protos, dim=1)


@torch.inference_mode()
def fit_logit_scale(protos, feats, labels, n=10000):
    """Pick a good temperature for cosine similarity, so the softmax of the prototype head is not too flat or too sharp"""
    idx = torch.randperm(len(feats), device=feats.device)[:n]
    cos = F.normalize(feats[idx].float(), dim=1) @ protos.T
    losses = {s: F.cross_entropy(cos * s, labels[idx]).item() for s in (5, 10, 15, 20, 30, 40, 60, 80)}
    return min(losses, key=losses.get)


def init_head(model, ctx, cfg):
    """Prototype head: each class's weight is the mean direction of its training features, times a temperature.
    The features are extracted at the size of the first epoch, TRAIN_SIZES[0].
    Returns the seconds spent, feature extraction included; they count toward this run's GPU time"""
    t0 = time.time()
    feats, labels = extract_features(model, ctx.train_items, cfg.TRAIN_SIZES[0], ctx, cfg)
    protos = class_prototypes(feats, labels, cfg.NUM_CLASSES)
    scale = fit_logit_scale(protos, feats, labels)
    with torch.no_grad():
        # The head works on unnormalized features: divide by the mean feature norm so logits ≈ scale × cosine similarity
        model.head.weight.copy_(protos * (scale / feats.float().norm(dim=1).mean()))
        model.head.bias.zero_()
    torch.cuda.synchronize()
    seconds = time.time() - t0
    log(f"Head initialized from the class prototypes of {len(feats):,} train images at {cfg.TRAIN_SIZES[0]} px "
        f"(logit scale {scale}) "
        f"in {seconds / 60:.1f} min, counted in this run's GPU time")
    return seconds

## 7. Training

- As in 01: AdamW, linear warmup for `WARMUP_EPOCHS` epochs and then a cosine decay to 0, layer-wise learning-rate decay (`build_param_groups`), bf16 mixed precision, gradient clipping at 1.0, label smoothing 0.1.
- New: epoch e trains at `TRAIN_SIZES[e]`. When the size changes, the training data loader is rebuilt with crops of the new size. The learning-rate schedule does not depend on the size.
- After each epoch, top-1 accuracy is measured on the holdout set at `EVAL_SIZE`.
- Training speed is measured at each training size, over 100 steps that start 20 steps after the size is set. It is written to the log and to `summary.json`.

In [ ]:
def build_param_groups(net, cfg):
    """Layer-wise LR decay: block i (1–12) gets LR × LAYER_DECAY^(13 − i); the patch embedding, class token and position
    embeddings count as layer 0, the final norm and the head get the full LR. No weight decay on 1-D params (biases,
    LayerNorm weights), the class token and the position embeddings"""
    num_layers = len(net.blocks) + 1
    groups = {}
    for n, p in net.named_parameters():
        m = re.match(r"blocks\.(\d+)\.", n)
        if m:
            layer = int(m.group(1)) + 1
        elif n.startswith(("cls_token", "pos_embed", "patch_embed")):
            layer = 0
        else:
            layer = num_layers
        lr = cfg.LR * cfg.LAYER_DECAY ** (num_layers - layer)
        wd = 0.0 if p.ndim <= 1 or n in ("cls_token", "pos_embed") else cfg.WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return [{"params": ps, "lr": lr, "weight_decay": wd, "base_lr": lr} for (lr, wd), ps in groups.items()]


@torch.inference_mode()
def evaluate(net, loader, ctx):
    """Top-1 and top-5 accuracy on a labeled set"""
    net.eval()
    c1 = c5 = n = 0
    for x, y in loader:
        with autocast():
            top5 = net(to_input(x, ctx)).topk(5, dim=1).indices
        y = y.to(DEVICE)
        c1 += (top5[:, 0] == y).sum().item()
        c5 += (top5 == y[:, None]).any(dim=1).sum().item()
        n += len(y)
    return c1 / n, c5 / n


def save_state(net, path, **extra):
    """Save the trained params: all of them, about 370 MB"""
    state = {n: p.detach().cpu() for n, p in net.named_parameters() if p.requires_grad or is_head(n)}
    torch.save({"state": state, **extra}, path)


def train(model, ctx, cfg, clock):
    """AdamW with linear warmup, then cosine decay to 0; bf16 autocast; gradient clipping at 1.0.
    Epoch e trains at TRAIN_SIZES[e] px; the training loader is rebuilt when the size changes. Evaluates on the holdout set
    at EVAL_SIZE after every epoch. Returns (history, optimizer steps, {training size: training speed in img/s})"""
    spe = len(ctx.train_items) // (cfg.BATCH_SIZE * cfg.ACCUM_STEPS)  # optimizer steps per epoch
    total, warmup = cfg.EPOCHS * spe, max(1, int(cfg.WARMUP_EPOCHS * spe))

    def lr_factor(s):
        if s < warmup:
            return (s + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total - warmup)))

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(build_param_groups(model, cfg), fused=True)
    train_loader = train_size = None
    holdout_loader = make_loader(ctx.holdout_items, False, cfg.EVAL_SIZE, ctx, cfg)
    hist = {"step": [], "loss": [], "epoch": [], "train_size": [], "epoch_loss": [], "holdout_top1": [], "steps_per_epoch": spe}
    log(f"Training {cfg.EPOCHS} epochs x {spe} steps (batch {cfg.BATCH_SIZE}x{cfg.ACCUM_STEPS}, warmup {warmup} steps)")
    step, speeds, t_mark, t_start = 0, {}, 0.0, clock.elapsed
    for epoch in range(cfg.EPOCHS):
        if cfg.TRAIN_SIZES[epoch] != train_size:
            train_size = cfg.TRAIN_SIZES[epoch]
            train_loader = None  # stop the worker processes of the old loader before starting new ones
            train_loader = make_loader(ctx.train_items, True, train_size, ctx, cfg)
            speed_start = step + 20  # measure the speed again at the new size
            log(f"Epoch {epoch + 1}: training at {train_size} px ({(train_size // 16) ** 2 + 1} tokens per image)")
        model.train()
        optimizer.zero_grad(set_to_none=True)
        run_loss, run_n, ep_loss, ep_n, t_log = 0.0, 0, 0.0, 0, time.time()
        with clock:
            for i, (x, y) in enumerate(train_loader):
                with autocast():
                    loss = F.cross_entropy(model(to_input(x, ctx)), y.to(DEVICE, non_blocking=True),
                                           label_smoothing=cfg.LABEL_SMOOTHING)
                (loss / cfg.ACCUM_STEPS).backward()
                run_loss += loss.detach()
                run_n += 1
                if (i + 1) % cfg.ACCUM_STEPS:
                    continue  # gradient accumulation: add up the gradients of ACCUM_STEPS batches before each update
                for g in optimizer.param_groups:
                    g["lr"] = g["base_lr"] * lr_factor(step)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                step += 1

                # Training speed at each size, measured over 100 steps that start 20 steps after the size is set
                # (after the data loader and the GPU have warmed up)
                if step == speed_start:
                    t_mark = clock.elapsed
                elif step == speed_start + 100:
                    sec_per_step = (clock.elapsed - t_mark) / 100
                    speeds[train_size] = cfg.BATCH_SIZE * cfg.ACCUM_STEPS / sec_per_step
                    log(f"Speed at {train_size} px: {speeds[train_size]:.0f} img/s, {sec_per_step * spe / 60:.1f} min/epoch")

                if step % cfg.LOG_EVERY == 0:
                    avg = (run_loss / run_n).item()
                    if not math.isfinite(avg):
                        raise RuntimeError("Loss is NaN/inf; try a lower LR")
                    eta = (total - step) * (clock.elapsed - t_start) / step
                    log(f"ep {epoch + 1}/{cfg.EPOCHS} step {step}/{total} | loss {avg:.4f} | "
                        f"lr {cfg.LR * lr_factor(step):.2e} | {run_n * cfg.BATCH_SIZE / (time.time() - t_log):.0f} img/s | "
                        f"{clock.elapsed / 60:.1f} min | ETA {eta / 60:.0f} min")
                    hist["step"].append(step)
                    hist["loss"].append(avg)
                    ep_loss += avg * run_n
                    ep_n += run_n
                    run_loss, run_n, t_log = 0.0, 0, time.time()
            if run_n:
                ep_loss += float(run_loss)
                ep_n += run_n
            top1 = evaluate(model, holdout_loader, ctx)[0]
        hist["epoch"].append(epoch + 1)
        hist["train_size"].append(train_size)
        hist["epoch_loss"].append(ep_loss / max(1, ep_n))
        hist["holdout_top1"].append(top1)
        log(f"== epoch {epoch + 1} (trained at {train_size} px): train loss {ep_loss / max(1, ep_n):.4f}, "
            f"holdout top-1 at {cfg.EVAL_SIZE} px {top1:.2%}, {clock.elapsed / 60:.1f} min used")
    return hist, step, speeds

## 8. Evaluation and inference latency

- After training, `resolution_sweep` evaluates the holdout set at every size in `FINAL_EVAL_SIZES`, and measures the inference latency and throughput at each size. As in 08, this sweep is not counted in the GPU time of the run.
- Inference latency is measured with random inputs. It only counts the forward pass of the model (not image loading or decoding), and uses bf16, the same as training. Batch size 1 is run 100 times and averaged; batch size 256 is run 20 times to get the throughput. The report must state the GPU type, batch size, precision and image size.

In [ ]:
INK, MUTED, GRID, SURFACE, LINE = "#0b0b0b", "#52514e", "#e1e0d9", "#fcfcfb", "#2a78d6"


def style_axis(ax, title, xlabel, ylabel):
    ax.set_facecolor(SURFACE)
    ax.set_title(title, loc="left", color=INK, fontsize=11)
    ax.set_xlabel(xlabel, color=MUTED)
    ax.set_ylabel(ylabel, color=MUTED)
    ax.grid(True, color=GRID, linewidth=0.75)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, length=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)


def plot_run_curves(hist, cfg, run_dir):
    """Save curves.png: the training loss, and the holdout top-1 after each epoch"""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), facecolor=SURFACE)
    axes[0].plot(hist["step"], hist["loss"], color=LINE, linewidth=1.5, solid_capstyle="round")
    style_axis(axes[0], f"Training loss ({METHOD})", "optimizer step", f"cross-entropy (label smoothing {cfg.LABEL_SMOOTHING})")
    acc = [100 * a for a in hist["holdout_top1"]]
    axes[1].plot(hist["epoch"], acc, color=LINE, linewidth=1.5, marker="o", markersize=6,
                 markeredgecolor=SURFACE, markeredgewidth=1.5)
    axes[1].annotate(f"{acc[-1]:.1f}%", (hist["epoch"][-1], acc[-1]), textcoords="offset points", xytext=(0, 8),
                     ha="center", color=INK, fontsize=9)
    axes[1].set_xticks(hist["epoch"], [f"{e}\n{s} px" for e, s in zip(hist["epoch"], hist["train_size"])])
    style_axis(axes[1], f"Holdout top-1 accuracy at {cfg.EVAL_SIZE} px", "epoch (training size)", "accuracy (%)")
    fig.tight_layout()
    fig.savefig(os.path.join(run_dir, "curves.png"), dpi=150, facecolor=SURFACE)
    plt.close(fig)


@torch.inference_mode()
def measure_latency(net, batch_size, iters, size, warmup=10):
    """Average seconds per forward pass on random size × size input: model compute only, no image reading or decoding"""
    net.eval()
    x = torch.randn(batch_size, 3, size, size, device=DEVICE)
    for _ in range(warmup):
        with autocast():
            net(x)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        with autocast():
            net(x)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters


def latency_report(net, size):
    lat_1 = measure_latency(net, 1, 100, size)
    lat_b = measure_latency(net, 256, 20, size)
    setting = f"{GPU_NAME}, bfloat16, no flip TTA"
    log(f"Inference latency at {size} px (batch 1): {lat_1 * 1e3:.2f} ms/img | throughput (batch 256): {256 / lat_b:.0f} img/s | "
        f"{setting}")
    return dict(latency_ms_bs1=lat_1 * 1e3, throughput_img_s=256 / lat_b, latency_setting=setting)


def resolution_sweep(net, ctx, cfg):
    """After training: holdout top-1/top-5, latency and throughput at each size of FINAL_EVAL_SIZES (EVAL_SIZE included)"""
    rows = []
    for size in sorted({*cfg.FINAL_EVAL_SIZES, cfg.EVAL_SIZE}):
        top1, top5 = evaluate(net, make_loader(ctx.holdout_items, False, size, ctx, cfg), ctx)
        log(f"Holdout at {size} px ({(size // 16) ** 2 + 1} tokens): top-1 {top1:.2%}, top-5 {top5:.2%}")
        rows.append(dict(size=size, tokens=(size // 16) ** 2 + 1, holdout_top1=top1, holdout_top5=top5,
                         **latency_report(net, size)))
    return rows

## 9. Full pipeline for one experiment

`run_experiment` runs these steps in order: build the model → initialize the head → apply the fine-tuning method → train → evaluate on the holdout set at every test size, with the inference latency at each size → write `summary.json`. The results of each experiment are saved on Drive in `cs604/runs/<run_name>/`: `train_log.txt`, `config.json`, `summary.json` (the result at each test size is in `by_size`), `history.json`, `curves.png`, and the weights `final.pt`.

In [ ]:
def run_experiment(overrides, run_name):
    """Run one experiment and return its summary. Everything is saved to cs604/runs/<run_name>/ on Drive"""
    global LOG_FILE
    cfg = resolve_config(overrides)
    run_dir = os.path.join(OUT_ROOT, "runs", run_name)
    os.makedirs(run_dir, exist_ok=True)
    LOG_FILE = os.path.join(run_dir, "train_log.txt")
    try:
        with open(os.path.join(run_dir, "config.json"), "w") as f:
            json.dump({**vars(cfg), "RUN_NAME": run_name, "overrides": overrides}, f, indent=2)
        log(f"\n========== {run_name} ==========")
        log(f"{METHOD}: lr {cfg.LR}, wd {cfg.WEIGHT_DECAY}, layer decay {cfg.LAYER_DECAY}, drop path {cfg.DROP_PATH}, epochs {cfg.EPOCHS}, "
            f"holdout {cfg.HOLDOUT_PER_CLASS}/class, prototype head")
        log(f"Image sizes: training {cfg.TRAIN_SIZES} px (one per epoch), holdout at {cfg.EVAL_SIZE} px after each epoch, "
            f"final sweep at {cfg.FINAL_EVAL_SIZES} px")
        log(f"Environment: {GPU_NAME} ({VRAM_GB} GB), bfloat16, {os.cpu_count()} CPUs | {VERSIONS}")
        random.seed(cfg.SEED)
        np.random.seed(cfg.SEED)
        torch.manual_seed(cfg.SEED)
        free_gpu()

        model = create_model(cfg)
        ctx = make_ctx(model, cfg)
        clock = Clock()
        clock.used += init_head(model, ctx, cfg)
        model = apply_method(model, cfg).to(DEVICE)
        params = param_report(model, ctx)
        log(f"Trainable (excl. head): {params['trainable_excl_head']:,} = {params['trainable_pct']:.3f}% of backbone | "
            f"head {ctx.n_head:,} params (fine-tuned)")

        hist, step, speeds = train(model, ctx, cfg, clock)
        save_state(model, os.path.join(run_dir, "final.pt"), config=vars(cfg))
        with open(os.path.join(run_dir, "history.json"), "w") as f:
            json.dump(hist, f)
        log(f"Training done: {step} steps, {clock.elapsed / 3600:.2f} GPU-hours")
        # The project doc asks for VRAM in the training log; reserved is closer to what nvidia-smi shows
        peak_gb, reserved_gb = torch.cuda.max_memory_allocated() / 1e9, torch.cuda.max_memory_reserved() / 1e9
        log(f"Peak VRAM: {peak_gb:.1f} GB allocated, {reserved_gb:.1f} GB reserved by PyTorch (GPU has {VRAM_GB} GB)")
        plot_run_curves(hist, cfg, run_dir)

        t0 = time.time()
        by_size = resolution_sweep(model, ctx, cfg)
        main = next(r for r in by_size if r["size"] == cfg.EVAL_SIZE)
        best = max(by_size, key=lambda r: r["holdout_top1"])
        log(f"Final holdout at {cfg.EVAL_SIZE} px: top-1 {main['holdout_top1']:.2%}, top-5 {main['holdout_top5']:.2%} | "
            f"best test size {best['size']} px: {best['holdout_top1']:.2%} | sweep took {(time.time() - t0) / 60:.1f} min "
            f"(not counted in the GPU time)")

        summary = dict(run=run_name, method=METHOD, model=cfg.MODEL_NAME, overrides=overrides, train_sizes=cfg.TRAIN_SIZES,
                       eval_size=cfg.EVAL_SIZE, **params, holdout_top1=main["holdout_top1"], holdout_top5=main["holdout_top5"],
                       best_size=best["size"], best_holdout_top1=best["holdout_top1"], by_size=by_size,
                       latency_ms_bs1=main["latency_ms_bs1"], throughput_img_s=main["throughput_img_s"],
                       latency_setting=main["latency_setting"], gpu=GPU_NAME, vram_gb=VRAM_GB,
                       batch=f"{cfg.BATCH_SIZE}x{cfg.ACCUM_STEPS}", epochs_trained=round(step / hist["steps_per_epoch"], 2),
                       steps=step, train_speed_img_s=speeds, gpu_hours=clock.elapsed / 3600,
                       peak_vram_gb=peak_gb, peak_vram_reserved_gb=reserved_gb, versions=VERSIONS)
        with open(os.path.join(run_dir, "summary.json"), "w") as f:
            json.dump(summary, f, indent=2)
        log(f"== {run_name} finished: holdout top-1 {main['holdout_top1']:.2%} at {cfg.EVAL_SIZE} px "
            f"({best['holdout_top1']:.2%} at {best['size']} px) | {summary['gpu_hours']:.2f} GPU-h")
        return summary
    finally:
        LOG_FILE = None

## 10. Run the experiments

Run each experiment in `EXPERIMENTS` in order. The run name is `<batch_name>_<n>_full_<sizes>`, for example `1106_0930_1_full_288` and `1106_0930_2_full_224to288`.

In [ ]:
def size_tag(sizes):
    """Short name of a size schedule for the run folder: [288, 288, 288] → '288', [224, 224, 288] → '224to288'"""
    return "to".join(str(s) for s in dict.fromkeys(sizes))


RESULTS = []
for i, exp in enumerate(EXPERIMENTS, 1):
    RESULTS.append(run_experiment(dict(exp), f"{BATCH_NAME}_{i}_{METHOD}_{size_tag(exp.get('TRAIN_SIZES', TRAIN_SIZES))}"))

## 11. Results tables and plot

Saved on Drive in `cs604/batches/<batch_name>/`:
- `results.md` (full data in `results.json`): one row per run, with the holdout result at `EVAL_SIZE`, the difference from the reference run at the same size, the best test size, the training cost, and the inference cost at `EVAL_SIZE`.
- `resolution.md`: the holdout top-1 of each run at every test size, next to the reference run `1019_1120_1_full` (trained at 224 px), plus the inference cost at each size.
- `resolution.png`: the same accuracies as a plot.

In [ ]:
from IPython.display import Markdown, display

# ---- results.md: one row per run ----
header = ("| Run | Training sizes (px) | Holdout top-1 | vs reference (points) | Holdout top-5 | Best test size | "
          "Train speed (img/s) | GPU-hours | Peak VRAM (GB) | Latency bs1 (ms) | Throughput (img/s) |\n"
          "|---|---|---|---|---|---|---|---|---|---|---|")
rows = []
for r in RESULTS:
    ref = REFERENCE.get(r["eval_size"])
    speed = ", ".join(f"{v:.0f} at {s} px" for s, v in r["train_speed_img_s"].items()) or "–"
    rows.append(f"| {r['run']} | {', '.join(map(str, r['train_sizes']))} | {r['holdout_top1']:.2%} at {r['eval_size']} px | "
                f"{'–' if ref is None else format(100 * (r['holdout_top1'] - ref), '+.2f')} | {r['holdout_top5']:.2%} | "
                f"{r['best_holdout_top1']:.2%} at {r['best_size']} px | {speed} | {r['gpu_hours']:.2f} | "
                f"{r['peak_vram_gb']:.1f} | {r['latency_ms_bs1']:.2f} | {r['throughput_img_s']:.0f} |")
table = (header + "\n" + "\n".join(rows) + f"\n\nvs reference: points above {REFERENCE_RUN} (trained at 224 px) tested at the "
         "same size. Latency and throughput are measured at the run's holdout size.")
with open(os.path.join(BATCH_DIR, "results.md"), "w") as f:
    f.write(table + "\n")
with open(os.path.join(BATCH_DIR, "results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2)
log("\n" + table)
display(Markdown(table))

# ---- resolution.md: holdout top-1 of each run at each test size, next to the reference ----
sizes = sorted({*REFERENCE, *(x["size"] for r in RESULTS for x in r["by_size"])})


def size_cells(acc):
    """One cell per test size; the best size of the row in bold"""
    best = max(acc.values())
    return " | ".join("–" if s not in acc else f"**{acc[s]:.2%}**" if acc[s] == best else f"{acc[s]:.2%}" for s in sizes)


cost = {x["size"]: x for x in RESULTS[0]["by_size"]}  # all runs have the same architecture: the cost depends only on the size
res_rows = [f"| Run | Training sizes (px) | {' | '.join(f'{s} px' for s in sizes)} |", "|---|---|" + "---|" * len(sizes),
            f"| {REFERENCE_RUN} (from 08) | 224, 224, 224 | {size_cells(REFERENCE)} |"]
res_rows += [f"| {r['run']} | {', '.join(map(str, r['train_sizes']))} | "
             f"{size_cells({x['size']: x['holdout_top1'] for x in r['by_size']})} |" for r in RESULTS]
res_rows += [f"| Tokens per image | | {' | '.join(str((s // 16) ** 2 + 1) for s in sizes)} |",
             f"| Latency bs1 (ms) | | {' | '.join(format(cost[s]['latency_ms_bs1'], '.2f') if s in cost else '–' for s in sizes)} |",
             f"| Throughput bs256 (img/s) | | "
             f"{' | '.join(format(cost[s]['throughput_img_s'], '.0f') if s in cost else '–' for s in sizes)} |"]
res_table = ("\n".join(res_rows) + "\n\nHoldout top-1 on the same 20,000 images; bold = the best test size of each run. "
             f"Latency and throughput: {GPU_NAME}, bfloat16, forward pass only.")
with open(os.path.join(BATCH_DIR, "resolution.md"), "w") as f:
    f.write(res_table + "\n")
log("\n" + res_table)
display(Markdown(res_table))

# ---- resolution.png: holdout top-1 against the test size ----
fig, ax = plt.subplots(figsize=(7.5, 4.2), facecolor=SURFACE)
ax.plot(sorted(REFERENCE), [100 * REFERENCE[s] for s in sorted(REFERENCE)], color=MUTED, linewidth=1.5, linestyle="--",
        marker="o", markersize=5, label=f"{REFERENCE_RUN} (trained at 224/224/224 px, from 08)")
for k, (r, color, marker) in enumerate(zip(RESULTS, ["#2a78d6", "#eb6834", "#1b9e77", "#7d5ba6"], "osD^")):
    ax.plot([x["size"] for x in r["by_size"]], [100 * x["holdout_top1"] for x in r["by_size"]], color=color, linewidth=2,
            marker=marker, markersize=6, markeredgecolor=SURFACE, markeredgewidth=1.5,
            label=f"{r['run']} (trained at {'/'.join(map(str, r['train_sizes']))} px)")
    ax.annotate(f"{100 * r['best_holdout_top1']:.2f}%", (r["best_size"], 100 * r["best_holdout_top1"]),
                textcoords="offset points", xytext=(0, 9 if k % 2 == 0 else -16), ha="center", color=INK, fontsize=9)
ax.set_xticks(sizes)
style_axis(ax, "Holdout top-1 against the test image size", "test image size (px)", "holdout top-1 (%)")
ax.legend(frameon=False, fontsize=8, loc="lower center")
fig.tight_layout()
fig.savefig(os.path.join(BATCH_DIR, "resolution.png"), dpi=150, facecolor=SURFACE)
display(fig)
plt.close(fig)

## 12. Auto disconnect (optional)

All results are already saved on Drive. `DISCONNECT_WHEN_DONE = True` releases the runtime after checking that everything is written, so it stops using compute units. Leave it off if you still want to run other experiments in the same session.

In [ ]:
if DISCONNECT_WHEN_DONE:
    from google.colab import runtime
    log("All done; releasing the runtime ...")
    drive.flush_and_unmount()  # make sure all results are written to Drive
    runtime.unassign()